# Description

This notebook is used to generate the figures in Supplementary Material: Dynamics in the $\{0,1\}$ Space. 

In [ ]:
# Packages
display("text/html", "<style>.container { width:100% !important; }</style>")

# Julia dependencies for plotting, stats, and experiments
using Plots, LaTeXStrings, Distributions, DoubleFloats, Plots.PlotMeasures, LinearAlgebra,
    DataFrames, CSV, LsqFit, StatsBase, Distances, Parameters, Random, StatsBase, Graphs, 
    GraphPlot, Base.Threads, Printf, JLD2

const ROOT_DIR = normpath(joinpath(pwd(), "..", ".."))
const SRC_DIR = joinpath(ROOT_DIR, "src")
const SCRATCH_DIR = mktempdir()

# Local modules
include(joinpath(SRC_DIR, "wagner_algorithm.jl"))
using ..BooleanNetwork: SimulationParameters, SimulationData
using ..BooleanNetwork: run_simulation
include(joinpath(SRC_DIR, "../src/data_processing.jl"))
using ..CustomStats: summarize_history, summarize_simulation_run, alignment_score, compute_mut_robustness

# Utilities
using ProgressMeter
using Base.Threads: Atomic, atomic_add!



In [ ]:
# Plotting utilities
const ROOT_DIR = normpath(joinpath(pwd(), ".."))
const FIGURES_DIR = joinpath(ROOT_DIR, "figures")
mkpath(FIGURES_DIR)
const ROBUSTNESS_FIG_DIR = joinpath(FIGURES_DIR, "scatterplots_mutational_robustness")
mkpath(ROBUSTNESS_FIG_DIR)

default(dpi=900, legendfontsize=24, guidefont=font(24), tickfont=font(24), margin=8Plots.mm, grid=false)

"""
    save_plot(plot_obj, filename)

Save a figure to the figures directory and print the output path.
"""
function save_plot(plot_obj, filename)
    filepath = joinpath(FIGURES_DIR, filename)
    savefig(plot_obj, filepath)
    println("Saved $(filepath)")
end
Plots.theme(:dao)

In [ ]:
function compare_histograms(
    data_vecs, labels;
    bins=30, xlabel="", ylabel="", title="", c_pal=:blues, normalize=:pdf)

    p1 = plot()

    colors = palette(c_pal, size(data_vecs, 1))

    for (idx, data_vec) in enumerate(collect(eachrow(data_vecs)))
        histogram!(p1, data_vec,
            bins=bins,
            xlabel=xlabel,
            ylabel=ylabel,
            label=labels[idx],
            title=title,
            color=colors[idx],
            normalize=normalize,
            alpha=0.30
        )
    end

    return p1
end

function b_score(weight_matrix, optimal_phenotype)
    trans_opt = 2*optimal_phenotype .- 1
    total_matches = 0
    number_genes = size(weight_matrix)[1]
    row_normalized = weight_matrix ./ [sum(abs.(weight_matrix[i, :])) for i in 1:size(weight_matrix)[1]]
    raw_b_score = sum(row_normalized .* trans_opt)

    return raw_b_score / number_genes
end

# Optimal Structure

In [ ]:
alt_activation(x) = x< -0.5 ? 0.5 : 1.0
alt_distance(x::AbstractVector, y::AbstractVector) = mean(abs.(x - y))

simulation_parameters = SimulationParameters(
    generations=200,
    activation_regime=(0.0,1.0), 
    activation_function=alt_activation,
    distance_function=alt_distance
    )

noiseless_results = run_simulation(simulation_parameters) # evolved without noise

simulation_parameters.noise_dist = Gamma(1.0, 1.0)
noisy_results = run_simulation(simulation_parameters)

simulation_parameters.noise_dist = Gamma(1.0/8.0, 8.0)
very_noisy_results = run_simulation(simulation_parameters)

In [ ]:
reds = cgrad(:reds)
purples = cgrad(:Purples)
axis_style = (xlabel="Generation", grid=false, majorgrid=false, minorgrid=false, legend=:best)

## B scores
b_noiseless = b_score.(noiseless_results.matrices_history, Ref(noiseless_results.optimal_phenotype))
b_noisy = b_score.(noisy_results.matrices_history, Ref(noisy_results.optimal_phenotype))
b_very_noisy = b_score.(very_noisy_results.matrices_history, Ref(very_noisy_results.optimal_phenotype))

p1 = plot(; ylabel="Rule Score " * L"B(𝐖)", axis_style...)

plot!(p1, vec(mean(b_noiseless, dims=2)), ribbon=2 .* vec(std(b_noiseless, dims=2)),
    label="Noiseless", color=:black, linestyle=:dot, linewidth=1.5, alpha=0.7)
plot!(p1, vec(mean(b_noisy, dims=2)), ribbon=2 .* vec(std(b_noisy, dims=2)),
    label="Noisy", color=purples[0.45])
plot!(p1, vec(mean(b_very_noisy, dims=2)), ribbon=2 .* vec(std(b_very_noisy, dims=2)),
    label="Very Noisy", color=purples[0.85])

## Alignment scores
a_noiseless = alignment_score.(noiseless_results.matrices_history, Ref(2 .* noiseless_results.optimal_phenotype .- 1))
a_noisy = alignment_score.(noisy_results.matrices_history, Ref(2 .* noisy_results.optimal_phenotype .- 1))
a_very_noisy = alignment_score.(very_noisy_results.matrices_history, Ref(2 .* very_noisy_results.optimal_phenotype .-1 ))

p2 = plot(; ylabel="Alignment Score " * L"A(𝐖)", axis_style...)

plot!(p2, vec(mean(a_noiseless, dims=2)), ribbon=2 .* vec(std(a_noiseless, dims=2)),
    label="Noiseless", color=:black, linestyle=:dot, linewidth=1.5, alpha=0.7)
plot!(p2, vec(mean(a_noisy, dims=2)), ribbon=2 .* vec(std(a_noisy, dims=2)),
    label="Noisy", color=reds[0.45])
plot!(p2, vec(mean(a_very_noisy, dims=2)), ribbon=2 .* vec(std(a_very_noisy, dims=2)),
    label="Very Noisy", color=reds[0.85])

p_comb = plot(p1,p2; layout = (1,2), size = (1000, 400), margins = 5Plots.mm)
savefig(p_comb, "bscore_and_alignment.png")
display(p_comb)

In [ ]:
conditions = [("Noiseless", noiseless_results),
    ("Noisy", noisy_results),
    ("Very Noisy", very_noisy_results)]

gens = [1, 10, 200]   # 500 = last generation

# One shared symmetric color scale across all conditions
m = maximum(maximum(abs, res.matrices_history[g, 1])
            for (_, res) in conditions for g in gens)

mat_style = (c=:redsblues, clims=(-m, m), yflip=true,
    aspect_ratio=:equal, framestyle=:box,
    ticks=false, colorbar=false)

panels = []
for (i, (name, res)) in enumerate(conditions)
    # Row label panel: only a rotated text
    plab = plot(framestyle=:none, legend=false, grid=false,
        xlims=(0, 1), ylims=(0, 1))
    annotate!(plab, 0.5, 0.5, text(name, 14, "Computer Modern",:center, rotation=90))
    push!(panels, plab)

    # Matrices at the chosen generations
    for g in gens
        push!(panels, heatmap(res.matrices_history[g, 1]; mat_style...,
            title=i == 1 ? "Generation $g" : ""))
    end

    # Optimal phenotype of this condition: same colormap, range fixed to [0, 1]
    v = reshape(res.optimal_phenotype, :, 1)
    pv = heatmap(v; c=:redsblues, clims=(0, 1), yflip=true,
        aspect_ratio=:equal, colorbar=false,
        framestyle=:box, ticks=false,
        title=i == 1 ? "Optimal" : "")
    hline!(pv, 0.5:1:(size(v, 1)+0.5), color=:white, linewidth=1, label=false)
    push!(panels, pv)
end

# Dedicated colorbar panel
pcb = scatter([NaN], [NaN]; zcolor=[0.0], c=:redsblues, clims=(-m, m),
    colorbar=true, framestyle=:none, legend=false,
    markerstrokewidth=0, markersize=0,
    colorbar_title="Weight " * L"W_{ij}",
    xlims=(0, 1), ylims=(0, 1))

l = @layout [[a{0.05w} b c d e{0.1w};
    f{0.05w} g h i j{0.1w};
    k{0.05w} l m n o{0.1w}] p{0.05w}]

p_heatmaps = plot(panels..., pcb; layout=l, size=(1250, 900))
savefig(p_heatmaps, "heatmaps.png")
display(p_heatmaps)

In [ ]:
include(joinpath(SRC_DIR,"../scripts/run_experiments.jl"))
EH = ExperimentHandler

Unfortunately, we don't find network motifs to be relevant in this scenario. 

In [ ]:
function summarize_expectations(data::Dict{K,Vector{Float64}}) where {K}
    return Dict(k => (mean(v), std(v) / sqrt(length(v))) for (k, v) in data)
    
end

In [ ]:
count_ffls_noiseless = EH.compute_ffl_type_counts(noiseless_results.matrices_history[end,:], 2 * noiseless_results.optimal_phenotype .- 1)
count_ffls_noisy = EH.compute_ffl_type_counts(noisy_results.matrices_history[end,:], 2 * noisy_results.optimal_phenotype .- 1)
count_ffls_very_noisy = EH.compute_ffl_type_counts(very_noisy_results.matrices_history[end, :], 2 * very_noisy_results.optimal_phenotype .- 1);

In [ ]:
# Relative frequencies

rel_noiseless = count_ffls_noiseless[2]
summ_rel_noiseless = summarize_expectations(rel_noiseless)
rel_noisy = count_ffls_noisy[2]
summ_rel_noisy = summarize_expectations(rel_noisy)
rel_very_noisy = count_ffls_very_noisy[2]
summ_rel_very_noisy = summarize_expectations(rel_very_noisy);

In [ ]:
data_dicts = [summ_rel_noiseless, summ_rel_noisy, summ_rel_very_noisy]
labels = ["Noiseless", "Noisy", "Very Noisy"]
shapes = [:circle, :utriangle, :star5]
offsets = [-0.2, 0.0, 0.2]

label_order = ["Coherent Type 1 (C1)", "Coherent Type 2 (C2)",
    "Coherent Type 3 (C3)", "Coherent Type 4 (C4)",
    "Incoherent Type 1 (I1)", "Incoherent Type 2 (I2)",
    "Incoherent Type 3 (I3)", "Incoherent Type 4 (I4)"]

short = [match(r"\((.*)\)", k).captures[1] for k in label_order]
x = 1:length(label_order)
is_coh = startswith.(label_order, "Coherent")

groups = [(findall(is_coh), "#348ABD"),
    (findall(.!is_coh), "#bf2b54")]

p = plot(xlabel="Network Motif Type", ylabel="Average Relative Frequency",
    grid=false, minorgrid=false, majorgrid=false,
    xticks=(x, short), minorticks=false, xlims=(0.4, length(label_order) + 0.6))

for (d, shp, off) in zip(data_dicts, shapes, offsets)
    avg = [d[k][1] for k in label_order]
    se = [d[k][2] for k in label_order]
    for (idx, col) in groups
        scatter!(p, x[idx] .+ off, avg[idx]; yerror=se[idx], label=false,
            marker=shp, markersize=7, 
            color=col, markerstrokecolor=:black, markerstrokewidth=1.5)
    end
end

# Legend entries: shapes = condition, colors = group
for (lab, shp) in zip(labels, shapes)
    scatter!(p, [NaN], [NaN]; label=lab, marker=shp, markersize=6,
        color=:gray, markerstrokecolor=:black)
end
for (lab, col) in zip(["Coherent", "Incoherent"], ["#348ABD", "#bf2b54"])
    scatter!(p, [NaN], [NaN]; label=lab, marker=:circle, markersize=6,
        color=col, markerstrokecolor=:black, legend=:best)
end

savefig(p, "motif_counts_single_pop_bscore.png")
display(p)

# Mutational Robustness

We create a function that obtains the optimal structure of the networks to certain degree. It is similar to the one written in $\texttt{alignment\_score\_properties.ipynb}$

In [ ]:
function create_signs_matrix_opt(number_genes, target_vector)
    translated_vec = 2 * target_vector .- 1
    sign_mat = [translated_vec[i] for i in 1:number_genes, j in 1:number_genes]
    return sign_mat
end

function create_matrix_opt(weight_dist, number_genes, number_opt_weights, target_vector)
    sign_matrix = create_signs_matrix_opt(number_genes, target_vector)
    
    cartesian_indices = sample(CartesianIndices(sign_matrix), 
        number_genes^2-number_opt_weights; 
        replace=false)

    for cartesian_idx in cartesian_indices
        sign_matrix[cartesian_idx] = sign_matrix[cartesian_idx] * -1.0
    end

    weight_matrix = abs.(rand(weight_dist, (number_genes,number_genes)))
    return sign_matrix .* weight_matrix
end

function plot_histogram_metric(metric_matrix;
    bins=30, title="", colors=["lightsalmon", "coral2", "firebrick2"],
    xlabel="", ylabel="", labels=["y1", "y2", "y3"])

    p1 = histogram(metric_matrix[1, :], bins=bins, title=title, color=colors[1], alpha=0.4,
        xlabel=xlabel, ylabel=ylabel, label=labels[1], majorgrid=false, minorgrid=false, grid=false,
        linecolor=:black, lw=0.5)
    histogram!(p1, metric_matrix[2, :], bins=bins, color=colors[2], alpha=0.4, label=labels[2],
        linecolor=:black, lw=0.5)
    histogram!(p1, metric_matrix[3, :], bins=bins, color=colors[3], alpha=0.8, label=labels[3],
        linecolor=:black, lw=0.5, legend=:best)

    return p1
end;

In [ ]:
number_genes = 10
replicates = 200
weights_dist = Normal(0.0, 1.0)
target_vector = rand([0.0, 1.0], number_genes)
initial_state = rand([0.0, 1.0], number_genes)
opt_values = [Int(number_genes^2 / 2), Int(3 * number_genes^2 / 4), Int(number_genes^2)]

n_mutations = 50
n_noise_masks = 200
weight_dist = Normal(0.0,1.0)
noise_dist = Gamma(1.0, 1.0)
mutation_prob = 0.01

shift = Matrix{Float64}(undef, 3, replicates)
var = Matrix{Float64}(undef, 3, replicates)
prob_shift = Matrix{Float64}(undef, 3, replicates)
prob_var = Matrix{Float64}(undef, 3, replicates)

progress = Progress(replicates; desc="Replicates")
@threads for rep_idx in 1:replicates
    for (alig_idx, alig_num) in enumerate(opt_values)
        matrix = create_matrix_opt(weight_dist, number_genes, alig_num, target_vector)

        result = compute_mut_robustness(
            matrix, initial_state, n_mutations, n_noise_masks, noise_dist, mutation_prob;
            weights_dist=weights_dist, max_steps=100,
            activation_function=alt_activation,
            development_type=:asynchronous
        )

        shift[alig_idx, rep_idx] = result.stable_expression_shift
        var[alig_idx, rep_idx] = result.stable_expression_var
        prob_shift[alig_idx, rep_idx] = result.unstable_prob_shift
        prob_var[alig_idx, rep_idx] = result.unstable_prob_var
    end
    next!(progress)
end

In [ ]:
pink_steps = range(colorant"#F8CAE4", stop=colorant"#832D51", length=6)

In [ ]:
green_steps = range(colorant"#CFDD9D", stop=colorant"#447A5F", length=6)

In [ ]:
colors_top_1 = pink_steps[4:6]
colors_top_2 = pink_steps[1:3]
colors_bot_1 = green_steps[4:6]
colors_bot_2 = green_steps[1:3]

shift_hist = plot_histogram_metric(
    shift, bins=range(minimum(shift), maximum(shift)*1.03, length=50),
    title="Stable expression shift", colors=colors_top_1, #[:pink, :hotpink2, :maroon3],
    xlabel=L"\Delta \bar{v}",
    ylabel="Frequency",
    labels=["$val / $(number_genes^2) optimal weights" for val in alignment_values]
)

expr_var_hist = plot_histogram_metric(
    var, bins=range(minimum(var), maximum(var)*1.03, length=50),
    title="Stable expression variance", colors=colors_top_2, #[:mediumorchid1, :mediumpurple2, :blueviolet],
    xlabel=L"\sigma^2",
    labels=["$val / $(number_genes^2) optimal weights" for val in alignment_values]
)
prob_shift_hist = plot_histogram_metric(
    prob_shift, bins=range(minimum(prob_shift), maximum(prob_shift)*1.03, length=50),
    title="Instability shift", colors=colors_bot_1, #[:deepskyblue, :dodgerblue2, :royalblue4],
    xlabel=L"\Delta p_u",
    ylabel="Frequency",
    labels=["$val / $(number_genes^2) optimal weights" for val in alignment_values]
)

prob_var_hist = plot_histogram_metric(
    prob_var, bins=range(minimum(prob_var), maximum(prob_var)*1.03, length=50),
    title="Instability variance", colors=colors_bot_2, #[:paleturquoise1, :turquoise1, :teal],
    xlabel=L"\rho^2",
    labels=["$val / $(number_genes^2) optimal weights" for val in alignment_values]
)

histograms = [shift_hist, expr_var_hist, prob_shift_hist, prob_var_hist]

combined_plot = plot(histograms..., layout=(2, 2), size=(900, 700))

savefig(combined_plot, "histograms_mut_b_score.png")
display(combined_plot)